In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nibabel as nib
from nibabel.affines import apply_affine
from scipy.ndimage import binary_erosion
from nilearn import datasets, image, plotting

ROOT = Path("/srv/lab/fmri/crave")
QC = ROOT / "derivatives" / "qc"
FL0 = ROOT / "derivatives" / "firstlevel" / "VS_bold-tedana_fwhm-0_chest-block_rev-6"   # unsmoothed (ROIs)
FL6 = ROOT / "derivatives" / "firstlevel" / "VS_bold-tedana_fwhm-6_chest-block_rev-6"   # 6 mm (whole brain)
ROI_DIR = ROOT / "derivatives" / "roi" / FL0.name
WB_DIR = ROOT / "derivatives" / "wholebrain" / FL6.name
ROI_DIR.mkdir(parents=True, exist_ok=True)
WB_DIR.mkdir(parents=True, exist_ok=True)

covariates = pd.read_csv(ROOT / "participants_covariates.csv", dtype={"sub": str})
run_qc = pd.read_csv(QC / "run_qc.csv")
summary = pd.read_csv(FL0 / "firstlevel_summary.csv")

CELLS = {"FixedDrop-FixedMagnitude": "FF", "FixedDrop-VariableMagnitude": "FV",
         "VariableDrop-FixedMagnitude": "VF", "VariableDrop-VariableMagnitude": "VV"}

## 1. ROI masks

Harvard-Oxford max-probability atlases (25%), resampled to the first-level grid.

| System | ROI | Definition |
|---|---|---|
| reward | `vs_nacc` | left + right accumbens |
| reward | `ofc` | frontal orbital cortex |
| reward | `ant_ins` | insular cortex, MNI y > 0 |
| control | `dlpfc` | 10 mm spheres at (±44, 36, 20) |
| control | `dacc` | anterior cingulate, MNI z > 15 |
| negative control | `wm`, `csf`, `v1`, `motor`, `global` | white matter (eroded), lateral ventricles (eroded), V1, precentral gyrus, all brain voxels |

In [ ]:
ref = nib.load(str(sorted(FL0.glob("sub-*/*_stat-effect.nii.gz"))[0]))

def atlas(kind):
    a = datasets.fetch_atlas_harvard_oxford(f"{kind}-maxprob-thr25-2mm")
    img = image.resample_to_img(a.maps, ref, interpolation="nearest")
    return np.asarray(img.dataobj).astype(int), [str(l) for l in a.labels]

def region(data, labels, *names):
    ids = [i for i, l in enumerate(labels) if any(n in l for n in names)]
    return np.isin(data, ids)

cort, cort_labels = atlas("cort")
subc, subc_labels = atlas("sub")
i, j, k = np.indices(ref.shape[:3])
xyz = apply_affine(ref.affine, np.stack([i, j, k], axis=-1))     # MNI coordinates of every voxel
dist_right = np.sqrt(((xyz - [44, 36, 20]) ** 2).sum(axis=-1))
dist_left = np.sqrt(((xyz - [-44, 36, 20]) ** 2).sum(axis=-1))

masks = {
    "vs_nacc": (region(subc, subc_labels, "Accumbens"), "reward"),
    "ofc": (region(cort, cort_labels, "Frontal Orbital Cortex"), "reward"),
    "ant_ins": (region(cort, cort_labels, "Insular Cortex") & (xyz[..., 1] > 0), "reward"),
    "dlpfc": ((dist_right <= 10) | (dist_left <= 10), "control"),
    "dacc": (region(cort, cort_labels, "Cingulate Gyrus, anterior division") & (xyz[..., 2] > 15), "control"),
    "wm": (binary_erosion(region(subc, subc_labels, "Cerebral White Matter"), iterations=2), "negctrl"),
    "csf": (binary_erosion(region(subc, subc_labels, "Lateral Ventric"), iterations=1), "negctrl"),
    "v1": (region(cort, cort_labels, "Intracalcarine", "Occipital Pole"), "negctrl"),
    "motor": (region(cort, cort_labels, "Precentral Gyrus"), "negctrl"),
}
for name, (m, system) in masks.items():
    print(f"{name:8s} {system:8s} {int(m.sum())} voxels")
    nib.Nifti1Image(m.astype(np.uint8), ref.affine).to_filename(str(ROI_DIR / f"roi-{name}.nii.gz"))

In [ ]:
# picture of the five a priori ROIs
labels = np.zeros(ref.shape[:3], np.int16)
for n, name in enumerate(["vs_nacc", "ofc", "ant_ins", "dlpfc", "dacc"], start=1):
    labels[masks[name][0]] = n
plotting.plot_roi(nib.Nifti1Image(labels, ref.affine), display_mode="ortho", cut_coords=(10, 20, 10),
                  title="1 VS/NAcc  2 OFC  3 anterior insula  4 dlPFC  5 dACC")
plt.savefig(ROI_DIR / "roi_masks.png", dpi=150)
plt.show()

## 2. ROI values per run

For each unsmoothed contrast map: mean estimate in each ROI. Voxels whose variance is more than 20 × the map's
median are dropped first (edge / signal-dropout voxels with exploding values). `coverage` is the share of the ROI kept.

In [ ]:
rows = []
for f in sorted(FL0.glob("sub-*/*_stat-effect.nii.gz")):
    sub = f.name.split("_")[0]
    run = f.name.split("task-")[1].split("_contrast")[0]                 # e.g. VS_run-01
    contrast = f.name.split("contrast-")[1].replace("_stat-effect.nii.gz", "")
    effect = nib.load(str(f)).get_fdata()
    variance = nib.load(str(f).replace("stat-effect", "stat-variance")).get_fdata()

    good = np.isfinite(effect) & (effect != 0)
    good = good & (variance <= 20 * np.median(variance[good]))

    rows.append({"sub": sub, "run": run, "contrast": contrast, "roi": "global", "system": "negctrl",
                 "beta": effect[good].mean(), "coverage": 1.0})
    for name, (m, system) in masks.items():
        keep = m & good
        rows.append({"sub": sub, "run": run, "contrast": contrast, "roi": name, "system": system,
                     "beta": effect[keep].mean(), "coverage": keep.sum() / m.sum()})

roi = pd.DataFrame(rows)
print(len(roi), "rows")

In [ ]:
# add condition, usable events, motion and participant covariates
roi = roi.merge(summary[["sub", "run", "condition"]], on=["sub", "run"], how="left")
roi["drop_schedule"] = roi["condition"].str.split("Drop").str[0]                       # Fixed / Variable
roi["magnitude_schedule"] = roi["condition"].str.split("-").str[1].str.replace("Magnitude", "")

usable = summary.set_index(["sub", "run"])
def n_usable(row):
    s = usable.loc[(row["sub"], row["run"])]
    if row["contrast"] == "reveal_all":
        return s[["usable_reveal_1", "usable_reveal_3", "usable_reveal_5"]].sum()
    if row["contrast"] == "anticipation_vs_boss":
        return min(s["usable_anticipation"], s["usable_boss_kill"])
    return s.get("usable_" + row["contrast"], np.nan)
roi["n_usable"] = roi.apply(n_usable, axis=1)

fd = run_qc[["sub", "run_label", "mean_fd_filt"]].rename(columns={"run_label": "run"})
roi = roi.merge(fd, on=["sub", "run"], how="left")
roi = roi.merge(covariates, on="sub", how="left")
roi["task"] = "VS"
roi.to_csv(ROI_DIR / "roi_values_long.csv", index=False)

print("participants:", roi["sub"].nunique(), "| runs:", roi.groupby(["sub", "run"]).ngroups)
roi.groupby("roi")["coverage"].median().round(2)

## 3. Participant maps for the whole-brain analysis

For each participant and event: average the 6 mm run maps within each condition cell, then compute

- `main` = mean of the cells
- `drop` = mean(VF, VV) − mean(FF, FV) — 60% vs 100% drop rate
- `mag` = mean(FV, VV) − mean(FF, VF) — variable vs fixed chest size
- `dxm` = (VV − VF) − (FV − FF) — needs all four cells
- `pmx` = mean(FF, VV) − mean(FV, VF) — pure vs mixed rounds

In fixed-size rounds every reveal has 3 items, so `reveal_all` uses the reveal-3 map there.

In [ ]:
COVARIATES = ["male", "gaming_log", "fd"]       # use ["fd"] for the version without sex and gaming time
EVENTS = ["anticipation", "reveal_all", "boss_kill"]

runs = roi[(roi["roi"] == "vs_nacc") & roi["condition"].isin(CELLS)].copy()
runs["cell"] = runs["condition"].map(CELLS)

people = covariates.set_index("sub")
people["grp"] = people["group"].map({"typical": 0, "at_risk": 1})
people["male"] = (people["sex"] == "male").astype(float)
people["gaming_log"] = np.log1p(people["gaming_h_per_day"]).fillna(np.log1p(people["gaming_h_per_day"]).median())
people["fd"] = runs.drop_duplicates(["sub", "run"]).groupby("sub")["mean_fd_filt"].mean()
people["risk_z"] = (people["gas7_mean"] - people["gas7_mean"].mean()) / people["gas7_mean"].std()

def contrast_maps(cells):
    m = lambda names: np.mean([cells[n] for n in names if n in cells], axis=0)
    out = {"main": np.mean(list(cells.values()), axis=0)}
    if ({"VF", "VV"} & cells.keys()) and ({"FF", "FV"} & cells.keys()):
        out["drop"] = m(["VF", "VV"]) - m(["FF", "FV"])
    if ({"FV", "VV"} & cells.keys()) and ({"FF", "VF"} & cells.keys()):
        out["mag"] = m(["FV", "VV"]) - m(["FF", "VF"])
    if {"FF", "FV", "VF", "VV"} <= cells.keys():
        out["dxm"] = (cells["VV"] - cells["VF"]) - (cells["FV"] - cells["FF"])
    if ({"FF", "VV"} & cells.keys()) and ({"FV", "VF"} & cells.keys()):
        out["pmx"] = m(["FF", "VV"]) - m(["FV", "VF"])
    return out

In [ ]:
maps = {}          # (event, effect) -> {sub: 3D array}
for event in EVENTS:
    if event == "reveal_all":
        # variable-size rounds have a reveal_all map; fixed-size rounds use reveal_3
        with_all = runs[runs["contrast"] == "reveal_all"]
        r3 = runs[runs["contrast"] == "reveal_3"]
        r3 = r3[~r3.set_index(["sub", "run"]).index.isin(with_all.set_index(["sub", "run"]).index)]
        ev_runs = pd.concat([with_all, r3])
    else:
        ev_runs = runs[runs["contrast"] == event]
    ev_runs = ev_runs[ev_runs["n_usable"] >= 1]
    for sub, g in ev_runs.groupby("sub"):
        cells = {}
        for cell, gc in g.groupby("cell"):
            files = []
            for run in gc["run"]:
                f = FL6 / sub / f"{sub}_ses-01_task-{run}_contrast-{event}_stat-effect.nii.gz"
                if not f.exists():                                   # fixed-size round: reveal_all = reveal_3
                    f = FL6 / sub / f"{sub}_ses-01_task-{run}_contrast-reveal_3_stat-effect.nii.gz"
                files.append(nib.load(str(f)).get_fdata())
            cells[cell] = np.mean(files, axis=0)
        for effect, arr in contrast_maps(cells).items():
            maps.setdefault((event, effect), {})[sub] = arr
    print(event, "done")

In [ ]:
# mask: voxels with data in every participant, inside the MNI grey-matter mask
data_mask = np.all([arr != 0 for arr in maps[("anticipation", "main")].values()], axis=0)
gm = image.resample_to_img(datasets.load_mni152_gm_mask(resolution=2), ref, interpolation="nearest")
mask = data_mask & (np.asarray(gm.dataobj) > 0)
nib.Nifti1Image(mask.astype(np.uint8), ref.affine).to_filename(str(WB_DIR / "mask.nii.gz"))
print(int(mask.sum()), "voxels in the mask")

In [ ]:
def write_vest(path, M, kind):
    M = np.atleast_2d(M)
    with open(path, "w") as f:
        f.write(f"/NumWaves\t{M.shape[1]}\n/{kind}\t{M.shape[0]}\n/Matrix\n")
        np.savetxt(f, M, fmt="%.6f", delimiter="\t")

commands = []
for (event, effect), by_sub in maps.items():
    folder = WB_DIR / f"{event}_{effect}"
    folder.mkdir(exist_ok=True)
    subs = sorted(by_sub)
    nib.Nifti1Image(np.stack([by_sub[s] for s in subs], axis=-1).astype(np.float32), ref.affine).to_filename(
        str(folder / "all_maps.nii.gz"))

    P = people.loc[subs]
    P[["grp", "risk_z"] + COVARIATES].to_csv(folder / "subjects.csv")
    cov = (P[COVARIATES] - P[COVARIATES].mean()).to_numpy()
    zeros = np.zeros(len(COVARIATES))
    designs = {
        "onesample": (np.column_stack([np.ones(len(P)), cov]), [np.r_[1, zeros], np.r_[-1, zeros]]),
        "group": (np.column_stack([P["grp"] == 1, P["grp"] == 0, cov]).astype(float), [np.r_[1, -1, zeros], np.r_[-1, 1, zeros]]),
        "risk": (np.column_stack([np.ones(len(P)), P["risk_z"] - P["risk_z"].mean(), cov]), [np.r_[0, 1, zeros], np.r_[0, -1, zeros]]),
    }
    for name, (X, C) in designs.items():
        write_vest(folder / f"design_{name}.mat", X, "NumPoints")
        write_vest(folder / f"design_{name}.con", np.array(C), "NumContrasts")
        commands.append(f"randomise -i {folder}/all_maps.nii.gz -o {folder}/rand_{name} -m {WB_DIR}/mask.nii.gz "
                        f"-d {folder}/design_{name}.mat -t {folder}/design_{name}.con -T -n 5000")
    print(f"{event:13s} {effect:5s} n = {len(subs)}")

(WB_DIR / "run_randomise.sh").write_text("\n".join(commands) + "\n")
print(len(commands), "randomise commands written")